# Урок 16. Резерв: «Побудь моделью» — версия 2.0

**Дата:** 07.01.2027 · **40–60 минут** · 8–9 класс  
**Модуль 3.** Данные — резервное занятие  
**Опирается на:** урок 1 (упражнение «Побудь моделью»), уроки 5 и 7 (циклы, функции), уроки 10–12 (pandas, разведка). Новых тем нет.

> **Для преподавателя.** 07.01 чаще всего каникулы. Ноутбук можно пройти дома. Он готовит к модулю «Машинное обучение», но не обязателен для понимания урока 17.

На уроке 1 ты искал правило «если часов не меньше …, то сдаст» **на глаз** по восьми ученикам. Сегодня поручим это компьютеру: напишем программу, которая **перебирает пороги** и выбирает лучший. А потом проверим найденное правило на **других** учениках. Всё — только тем, что мы уже умеем.

## Цели

1. Автоматизировать поиск правила по примерам с помощью цикла.
2. Посчитать, какую долю учеников правило угадывает.
3. Проверить правило на новых данных и сравнить результат.
4. Повторить pandas и функции перед вторым полугодием.

## 1. Данные

Та же функция-генератор, что на уроках 12–13. Сначала возьмём **40 учеников** — по ним будем искать правило.

In [ ]:
import random
import pandas as pd

def make_students(n, seed):
    """Сгенерировать учебную таблицу учеников (как на уроке 12)."""
    random.seed(seed)
    rows = []
    for i in range(n):
        hours = random.randint(1, 14)
        attendance = min(100, max(20, round(50 + 3 * hours + random.gauss(0, 10))))
        score = min(100, max(0, round(20 + 4 * hours + 0.3 * attendance + random.gauss(0, 8))))
        rows.append({
            "класс": random.choice([8, 9]),
            "кружок": random.choice(["Python", "Роботы", "Игры"]),
            "часы": hours,
            "посещаемость": attendance,
            "сон": round(random.uniform(6, 9.5), 1),
            "балл": score,
        })
    return pd.DataFrame(rows)

students = make_students(40, seed=7)
students["сдал"] = students["балл"] >= 60
students.head()

## 2. Правило и его точность

Правило: «если часов не меньше `t`, то сдаст». Для каждого ученика правило даёт **ответ** (`True / False`), а в таблице есть **настоящий результат** — столбец `сдал`. Доля совпадений — это **точность** правила.

In [ ]:
def rule_accuracy(table, t):
    """Доля учеников, для которых правило «часы >= t» угадало результат."""
    guess = table["часы"] >= t
    correct = guess == table["сдал"]
    return correct.mean()

print("Порог 3:", round(rule_accuracy(students, 3), 2))
print("Порог 7:", round(rule_accuracy(students, 7), 2))
print("Порог 12:", round(rule_accuracy(students, 12), 2))

`guess == table["сдал"]` сравнивает два столбца построчно: `True`, где ответ правила совпал с настоящим. Среднее от столбца `True / False` — доля `True` (урок 12).

## 3. Пусть компьютер найдёт лучший порог

Переберём все пороги от 1 до 14 и запомним лучший — паттерн «максимум» из урока 5.

In [ ]:
best_t = 1
best_acc = 0
for t in range(1, 15):
    acc = rule_accuracy(students, t)
    print(f"порог {t:2}: точность {acc:.2f}")
    if acc > best_acc:
        best_acc = acc
        best_t = t

print()
print("Лучший порог:", best_t, "| точность:", round(best_acc, 2))

Только что компьютер **научился** правилу по примерам: мы не говорили ему порог — он нашёл его сам, перебрав варианты. Это и есть главная идея машинного обучения из урока 1, в самом простом виде.

## 4. Главный вопрос: а на новых учениках?

Правило подбирали по этим 40 ученикам. Но нужно оно для **новых** — тех, кого мы ещё не видели. Сгенерируем другую группу (другое зерно) и проверим найденный порог на ней, **ничего не меняя**.

In [ ]:
new_students = make_students(40, seed=2027)
new_students["сдал"] = new_students["балл"] >= 60

print("Точность на «своих» учениках:", round(rule_accuracy(students, best_t), 2))
print("Точность на новых учениках:  ", round(rule_accuracy(new_students, best_t), 2))

Обычно на новых учениках точность **немного ниже**: правило подстроилось под особенности тех 40, по которым его искали. Насколько ниже — зависит от удачи и от количества данных.

**Подумай:**

1. Почему честно оценивать правило нужно на учениках, которых **не было** при поиске порога?
2. Что будет, если искать порог по 8 ученикам, как на уроке 1? Попробуй в разделе 5.
3. Можно ли сделать правило точнее, если учитывать ещё и посещаемость?

Это ровно те вопросы, с которых начнётся модуль «Машинное обучение».

## 5. Эксперимент: мало данных

Повтори поиск порога на маленькой группе и проверь на большой новой.

In [ ]:
tiny = make_students(8, seed=16)
tiny["сдал"] = tiny["балл"] >= 60
# TODO: найди лучший порог на tiny циклом, как в разделе 3,
# и проверь его точность на new_students

<details>
<summary>Пример решения</summary>

```python
best_t_tiny = 1
best_acc_tiny = 0
for t in range(1, 15):
    acc = rule_accuracy(tiny, t)
    if acc > best_acc_tiny:
        best_acc_tiny = acc
        best_t_tiny = t

print("Порог по 8 ученикам:", best_t_tiny, "| точность на них:", round(best_acc_tiny, 2))
print("Точность этого порога на новых:", round(rule_accuracy(new_students, best_t_tiny), 2))
```
На маленькой группе легко найти правило, которое угадывает почти всех, — но на новых учениках оно часто работает хуже. Мало примеров → легко «подогнать» правило под случайности.
</details>

## 6. Готовность ко второму полугодию

- [ ] Colab открывается, ноутбуки сохраняются на Диск.
- [ ] Помню, как читать CSV в pandas и снимать паспорт.
- [ ] Могу почистить таблицу и объяснить решения.
- [ ] Умею сравнивать группы и строить графики с выводами.
- [ ] Могу объяснить, чем правила отличаются от машинного обучения.

**14.01.2027:** модуль «Машинное обучение» — учимся ставить задачу для модели.